In [46]:
import numpy as np
from LanzaModels import TIK_1D
from ADMMsRustici import SpectralSolver
from signalClass import *
import time

In [47]:
np.random.seed(24102000)
n = 1024

construct blur matrix

In [48]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [ ]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 100)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [50]:
mu = 2
VarModel = TIK_1D.TIK_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [51]:
#begin solver construction
np.random.seed(24102002)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = SpectralSolver.SpectralSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [52]:
iters = 30

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [53]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ (VarModel.Q @ (yk - yk_1)) )

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    ImgHistory[iter] = VarModel(xk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResidue) <= 1e-9):
        print(iter)
        break


1 / 30
2 / 30
3 / 30
4 / 30
5 / 30
6 / 30
7 / 30
8 / 30
9 / 30
10 / 30
11 / 30
12 / 30
13 / 30
14 / 30
15 / 30
16 / 30
17 / 30
18 / 30
19 / 30
20 / 30
21 / 30
22 / 30
23 / 30
24 / 30
25 / 30
26 / 30
27 / 30
28 / 30
29 / 30
30 / 30


In [54]:
np.savez_compressed(
    "./SpectralADMMTIK-Gauss.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)